In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/keras/gemma/keras/gemma_2b_en/3/config.json
/kaggle/input/models/keras/gemma/keras/gemma_2b_en/3/tokenizer.json
/kaggle/input/models/keras/gemma/keras/gemma_2b_en/3/metadata.json
/kaggle/input/models/keras/gemma/keras/gemma_2b_en/3/model.weights.h5
/kaggle/input/models/keras/gemma/keras/gemma_2b_en/3/assets/tokenizer/vocabulary.spm


In [23]:
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


# Set environment variables for Kaggle API

In [24]:
import os

# Replace these with your Kaggle credentials
os.environ["KAGGLE_USERNAME"] = "ankitayadav2312"
os.environ["KAGGLE_KEY"] = "KGAT_28bb39f8da459bf895cf376573d3f2a4"

# Set Backend (Important)

In [25]:
# Set the Keras backend to JAX (or PyTorch/TensorFlow)
os.environ["KERAS_BACKEND"] = "jax"
# Avoid memory fragmentation on JAX backend
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

In [26]:
import keras
import keras_nlp
import json
import numpy as np
keras.config.set_floatx("bfloat16")

# Download Dolly Dataset

In [27]:
!wget -O databricks-dolly-15k.jsonl \
https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

--2026-03-04 13:40:53--  https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl
Resolving huggingface.co (huggingface.co)... failed: Temporary failure in name resolution.
wget: unable to resolve host address ‘huggingface.co’


# Preprocess Dataset

In [28]:
import json
data = []

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        features = json.loads(line)

        # Skip examples with context
        if features["context"]:
            continue

        template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
        data.append(template.format(**features))

# Use only 1000 samples for faster training
data = data[:1000]

print("Total training samples:", len(data))

Total training samples: 0


# Load Gemma Model

In [29]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(
    "gemma_2b_en"
)

gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (4.67 GB)

 Trainable params: 2,506,172,416 (4.67 GB)

 Non-trainable params: 0 (0.00 B)

In [30]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)

print("--- RESPONSE BEFORE FINE-TUNING ---")
print(gemma_lm.generate(prompt, max_length=256))

--- RESPONSE BEFORE FINE-TUNING ---
Instruction:
What should I do on a trip to Europe?

Response:
The first thing to remember is to take your camera with you. Europe is full of beautiful scenery and interesting people. The next step is to plan a trip that is within your budget and that you will enjoy.

The next step is to book your flights and hotel accommodations. You will need to book these well in advance so that you can get the best prices. Once you have booked your flights and your hotel, you can start planning your trip.

The first thing you should do is to research the places that you would like to visit. Once you have a rough idea of what you would like to see, you can start planning your itinerary.

It is important to remember that Europe is a large continent and there are many different countries within it. Therefore, you may want to break up your trip into multiple trips, each focusing on a different part of Europe.

Once you have your itinerary planned, it is time to start 

# Enable LoRA

In [31]:
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,507,536,384 (4.67 GB)

 Trainable params: 1,363,968 (2.60 MB)

 Non-trainable params: 2,506,172,416 (4.67 GB)

# Limit input sequence length to control memory

In [32]:
gemma_lm.preprocessor.sequence_length = 512

# Compile the model

In [33]:
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

# Compile Model

In [34]:
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Train (LoRA Fine-tuning)

In [35]:
import tensorflow as tf

# 1. Create the dataset and force the type to string
# (Note: If this line crashes, it means there is a non-string, like a NaN/float, inside your 'data' list!)
train_dataset = tf.data.Dataset.from_tensor_slices(tf.constant(data, dtype=tf.string))
train_dataset = train_dataset.batch(1)

# 2. MANUALLY apply the Gemma preprocessor outside of the fit loop
# This converts the strings into token IDs (integers) before Keras can mess with them
train_dataset = train_dataset.map(gemma_lm.preprocessor, num_parallel_calls=tf.data.AUTOTUNE)

print("Starting fine-tuning...")

# 3. Fit on the pre-tokenized integer data
gemma_lm.fit(
    train_dataset,
    epochs=1,
)

Starting fine-tuning...


TypeError: in user code:

    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/models/task.py", line 67, in preprocess_samples  *
        return self.preprocessor(x, y=y, sample_weight=sample_weight)
    File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/utils/tensor_utils.py", line 69, in wrapper
        x = fn(self, x, y=y, sample_weight=sample_weight, **kwargs)
    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/models/causal_lm_preprocessor.py", line 97, in call
        x = self.tokenizer(x)
    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/utils/tensor_utils.py", line 57, in wrapper
        x = fn(self, x, **kwargs)
    File "/usr/local/lib/python3.12/dist-packages/keras_hub/src/tokenizers/tokenizer.py", line 197, in call
        return self.tokenize(inputs, *args, *

# Test After Fine-tuning

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="Explain photosynthesis like I am 5 years old.",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))

In [ ]:
sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print("Europe Trip Prompt:")
print(gemma_lm.generate(prompt, max_length=256))

In [ ]:
prompt = template.format(
    instruction="Explain the process of photosynthesis in a way that a child could understand.",
    response="",
)

In [ ]:
print("\nELI5 Photosynthesis Prompt:")
print(gemma_lm.generate(prompt, max_length=256))

# If Output Is Too Complex

In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(
    "gemma_1.1_instruct_2b_en"
)